In [1]:
# Identifiers (RQx, Hx, Px, Gx, Fx, Tx) are defined in vpir_bench_spec.md: section 2 for the research questions,
# section 11 for the tests behind each gate, section 12 for phases, gates and figures. Rx reproduction targets are
# in docs/reproduction_targets.md, and numbered decisions in plan.md.
# RQ1: do the published overhead ratios hold on common hardware? (spec 2, H1)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import frame, plots
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
t = plots.build(df, status)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


def save_table(name, table):
    (FIG / f"{name}.md").write_text(plots._markdown(table, plots.TABLE_TITLES[name], False, plots.TABLE_NOTES.get(name)),
                                   encoding="utf-8")


plots._style()
print(f"{len(df)} measurement rows from {', '.join(r.name for r in RUNS)}; {len(status)} (cell, phase) rows not ok")

3427 measurement rows from all-20261112_tier1, all-20261112_anchors; 14 (cell, phase) rows not ok


In [2]:
# T3: every target in configs/reproduction_targets.yaml against the grid and the anchors, with S4's extrapolated
# sizes for R8 (decision 10). H1: communication within +-5 % of the published value (rule exact; plot readings
# within a factor 1.2), compute within the published range up to a hardware factor (rule hw).
t3 = t.t3
rows = t3[t3.verdict != "see summary row"]
print(rows.verdict.value_counts().to_string(), "\n")
for rule in ("exact", "hw"):
    r = rows[rows.rule == rule]
    print(f"rule {rule}: {len(r)} rows, verdicts {r.verdict.value_counts().to_dict()}")
print()
print(t3[["id", "cell", "ours", "ci_lo", "ci_hi", "unit", "target", "deviation", "verdict"]].round(3).to_string(index=False))
save_table("T3_reproduction", t3.drop(columns=["check", "target_lo", "target_hi"]))

verdict
pass                            32
hw factor                       10
missing                          7
context (different baseline)     7
explain                          1 

rule exact: 25 rows, verdicts {'pass': 22, 'missing': 3}
rule hw: 32 rows, verdicts {'pass': 10, 'hw factor': 10, 'context (different baseline)': 7, 'missing': 4, 'explain': 1}

           id                                    cell       ours      ci_lo      ci_hi unit                    target  deviation                      verdict
      R1-time         M1 1MiB x 1KiB / M0 1MiB x 1KiB      1.217      1.139      1.314    x              max <= 2.9 x        NaN              see summary row
      R1-time       M1 16MiB x 1KiB / M0 16MiB x 1KiB      1.363      1.337      1.412    x              max <= 2.9 x        NaN              see summary row
      R1-time     M1 128MiB x 1KiB / M0 128MiB x 1KiB      1.479      1.454      1.492    x              max <= 2.9 x        NaN              see summary row
     

In [3]:
# The rows H1 does not pass, with what each needs: "explain" is outside tolerance on our hardware; "missing" has
# no row (a cell that ran out of memory, or a size we do not run).
odd = t3[t3.verdict.isin(["explain", "missing"])]
print(odd[["id", "claim", "cell", "ours", "target", "deviation", "verdict"]].round(3).to_string(index=False))
print("\ncells not ok in the runs:")
print(status.groupby(["arm", "status"]).size().to_string())

           id                                                             claim                                cell  ours        target  deviation verdict
R3-time-ratio                                          M1/M0 user time at 1 GiB     M1 1GiB x 1KiB / M0 1GiB x 1KiB 1.554         1.9 x     -0.182 explain
           R5                        S3 (LWE+) user time over S0-go (SimplePIR) S3 1GiB x 1 bit / S0-go 1GiB x 256B   NaN   [30, 100] x        NaN missing
   R9-S3-1GiB                                      S3 (LWE+) user time at 1 GiB                     S3 1GiB x 1 bit   NaN      54000 ms        NaN missing
  R10-S3-1GiB                                       S3 digest download at 1 GiB                     S3 1GiB x 1 bit   NaN    400000 KiB        NaN missing
  R11-S3-1GiB                                          S3 online bytes at 1 GiB                     S3 1GiB x 1 bit   NaN     11000 KiB        NaN missing
          R12                             S0-go hint smaller than the 

In [4]:
# F1: published against measured, per claim, with 95 % CIs.
fig = plots.f1_reproduction(t)
save("F1_reproduction", fig)
print(sorted(p.name for p in FIG.glob("F1_*")))

['F1_reproduction.pdf', 'F1_reproduction.png']


In [5]:
# Communication targets only: every measured byte count (KiB or MiB) against its published value.
b = rows[rows.unit.isin(["KiB", "MiB"]) & rows.ours.notna()]
dev = b.deviation.abs()
print(f"{len(b)} byte targets measured, max |deviation| {dev.max():.3f} ({b.loc[dev.idxmax(), 'id']}),",
      f"{(dev <= 0.03).sum()} within 3 %, {(dev <= 0.05).sum()} within 5 %")


16 byte targets measured, max |deviation| 0.192 (R11-S0-1MiB), 12 within 3 %, 14 within 5 %
